# Notebook 4: Spatial Querying

## What this notebook does

The Domain API understands suburb names. But urban research often works with ABS boundaries (SA2, LGA, Greater Capital City regions) that do not align neatly with suburb names. An SA2 may cover parts of several suburbs. A suburb may straddle two SA2 boundaries. Querying suburb by suburb introduces extra costs, gaps and overlaps.

The API's solution is `geoWindow`: you feed the API a polygon as a list of coordinates and ask for all listings whose property location falls inside it.

## Why spatial querying is the recommended approach

For any study area larger than a single suburb, spatial querying is almost always the most credit-efficient approach.

When querying suburb by suburb, every suburb costs at least one probe credit plus one credit per page of results. Ten suburbs means ten probes and ten separate sets of fetch pages. A polygon covering the same ten suburbs costs **one probe and one set of fetch pages for the entire area**, the same structure as a single-suburb query, regardless of how many suburb boundaries sit inside it.

The suburb-by-suburb approach from Notebook 3 is still useful when you specifically want per-suburb breakdowns, or when your study area is defined by a list of suburb names rather than a geographic boundary. For everything else, SA2s, LGAs, catchment areas, corridor studies, a polygon query is the right starting point.

This notebook teaches you:

1. What GeoJSON boundary files are and where the pre-prepared ones live.
2. Why the coordinate format in GeoJSON files must be converted before sending
   to the API.
3. The difference between a bounding box (rough rectangle) and a polygon
   (exact boundary), and when to use each.
4. How to run a spatial listing search and visualise the results on a map.
5. How cursor advancement (from Notebook 3) applies equally to spatial queries.

**By the end** you will have downloaded listings for the Fitzroy/Collingwood area
using a precise polygon boundary and produced a map showing where those listings sit.

---

## Before You Start

**If you have already completed Notebook 0**, your packages, `.env` file, and connection are already set up.

If this is your first time using Domain API, complete `notebook-0-getting-started.ipynb` first. It covers package installation, creating the `.env` credentials file, and making a test call to confirm your connection works. Then complete Notebook 1 before this one, as it introduces the listings endpoint used throughout this notebook. If you plan to query a large area (with more than 1,000 listings potentially), read Notebook 3 first for the cursor advancement technique.

### Set your boundary file path

This notebook uses a GeoJSON boundary file. Set the path to wherever the file lives on your machine in the setup cell below. If you are working from the AURIN training materials repository, an example file is provided in the `test_areas` folder, but the path is not assumed, set it explicitly.

Depending on the operating system you are using, copying and pasting a path directly may cause an error. Windows file manager gives you backslashes (`C:\Users\you\boundaries\area.geojson`), while Mac and Linux give you forward slashes (`/Users/you/boundaries/area.geojson`). A single backslash has a special meaning inside quotes in Python, so either swap the backslashes for forward slashes (`'C:/Users/you/boundaries/area.geojson'`), which works on every operating system, double each one (`'C:\\Users\\you\\boundaries\\area.geojson'`), or put an `r` in front of the quotes (`r'C:\Users\you\boundaries\area.geojson'`).

---
## Key Terms

**GeoJSON:** A standard file format for storing geographic shapes (points, lines, polygons)
as text. The ABS publishes its SA2, LGA, and other boundary files in this format.

**Polygon:** A closed shape defined by a list of coordinates (corner points). A suburb
boundary or SA2 boundary is stored as a polygon.

**Coordinate:** A pair of numbers that specifies a location on Earth. The first number
is longitude (east-west position) and the second is latitude (north-south position).
Melbourne is roughly at latitude -37.8 and longitude 144.9.

**Bounding box:** The smallest rectangle that fully encloses a polygon, defined by its
northernmost, southernmost, easternmost, and westernmost points. It is faster to compute
than the full polygon but captures a larger area.

**SA2 (Statistical Area Level 2):** An ABS geographic unit containing roughly 3,000 to
25,000 people, used for census and demographic data.

**geoWindow:** The parameter name in the Domain API for spatial filtering. It accepts
either a polygon or a bounding box.

---
## Setup: Check Packages and Files

Run the cell below to confirm all required packages are installed and your boundary file is found.

In [ ]:
# Check required packages
missing = []
for pkg in ['requests', 'pandas', 'matplotlib', 'dotenv', 'folium']:
    try:
        __import__(pkg)
    except ImportError:
        missing.append(pkg if pkg != 'dotenv' else 'python-dotenv')

if missing:
    print('Missing packages. Run in terminal:')
    print(f'  pip install {" ".join(missing)}')
else:
    print('Packages: OK')

# -----------------------------------------------------------------------
# Set the path to your GeoJSON boundary file here.
# This can be an absolute path or relative to the location of this notebook.
# Use forward slashes (/) on Mac and Linux. On Windows, use either forward
# slashes or doubled backslashes (\\), because a pasted Windows path with
# single backslashes will not work here.
# -----------------------------------------------------------------------
from pathlib import Path

FITZROY_COLLINGWOOD = Path('../../test_areas/fitzroy_collingwood.geojson')
# -----------------------------------------------------------------------

if FITZROY_COLLINGWOOD.exists():
    print(f'Fitzroy/Collingwood boundary: Found  ({FITZROY_COLLINGWOOD.resolve()})')
else:
    print(f'Fitzroy/Collingwood boundary: NOT FOUND')
    print(f'  Expected at: {FITZROY_COLLINGWOOD.resolve()}')
    print(f'  Update the path at the top of this cell.')

## Setup: Connect to the API

In [ ]:
import sys
import json
import pandas as pd
import matplotlib.pyplot as plt
import folium
from IPython.display import display

if '.' not in sys.path:
    sys.path.insert(0, '.')

from utils import (
    PROXY_BASE,
    APICallTracker,
    probe_count,
    geojson_file_to_api_polygon,
    geojson_bbox_to_api_box,
    validate_australia_coords,
)

tracker = APICallTracker()

import os
print(f'Connected to: {PROXY_BASE}')
print(f'Account:      {os.getenv("AURIN_USERNAME", "(not found; check your .env file)")}')

---
## Credit Comparison: Suburb-by-Suburb vs. Polygon

Before getting into the mechanics, it helps to see the credit saving in concrete numbers using the same 10 inner-Melbourne suburbs from Notebook 3.

The suburb-by-suburb approach sends one probe per suburb, 10 API calls. A single bounding box covering the same area costs 1 API call. The saving carries through to the full fetch too: each suburb also needs its own set of page calls, while a polygon retrieves the entire area in one continuous run.

**These two cells cost 11 credits total.** If you want to save credits, read the output and move on, you do not need to re-run them. If you are planning a large extraction, use the [Credit Calculator guide](../../phase-1/03-credit-calculator.md) to estimate costs before running.

In [ ]:
# --- Approach 1: one probe per suburb (10 credits) ---
SUBURBS_NB3 = [
    ('Carlton',     'VIC', '3053'),
    ('Fitzroy',     'VIC', '3065'),
    ('Collingwood', 'VIC', '3066'),
    ('Richmond',    'VIC', '3121'),
    ('Brunswick',   'VIC', '3056'),
    ('Northcote',   'VIC', '3070'),
    ('Abbotsford',  'VIC', '3067'),
    ('Prahran',     'VIC', '3181'),
    ('St Kilda',    'VIC', '3182'),
    ('South Yarra', 'VIC', '3141'),
]

START_DATE = '2022-01-01'
suburb_counts = {}
calls_before = tracker.total

for suburb, state, postcode in SUBURBS_NB3:
    payload = {
        'listingType': 'Sold',
        'listedSince': START_DATE,
        'locations': [{'state': state, 'suburb': suburb, 'postCode': postcode,
                       'includeSurroundingSuburbs': False}],
    }
    count = probe_count(payload, tracker)
    suburb_counts[suburb] = count or 0
    print(f'  {suburb:<15}  {count:>6,} records')

suburb_probe_calls = tracker.total - calls_before
suburb_total = sum(suburb_counts.values())
tracker.checkpoint('Credit comparison: suburb probes (10 calls)')

print(f'\nSuburb-by-suburb:  {suburb_probe_calls} probe credits  →  {suburb_total:,} records total')

# --- Approach 2: one bounding box covering the same area (1 credit) ---
# Approximate bounding box for the 10 inner-Melbourne suburbs above
inner_melb_box_payload = {
    'listingType': 'Sold',
    'listedSince': START_DATE,
    'geoWindow': {
        'box': {
            'topLeft':     {'lat': -37.755, 'lon': 144.940},
            'bottomRight': {'lat': -37.875, 'lon': 145.025},
        }
    },
}

calls_before = tracker.total
bbox_count = probe_count(inner_melb_box_payload, tracker)
bbox_probe_calls = tracker.total - calls_before
tracker.checkpoint('Credit comparison: bounding box probe (1 call)')

print(f'Single bounding box: {bbox_probe_calls} probe credit   →  {bbox_count:,} records total')
print()

savings = suburb_probe_calls - bbox_probe_calls
print(f'Probe credits saved: {savings} of {suburb_probe_calls} ({savings / suburb_probe_calls * 100:.0f}% reduction)')
print()
print('Note: the bounding box count may be slightly higher because the rectangle')
print('extends a little beyond some suburb edges, capturing neighbouring properties.')

---
## What Is a GeoJSON File?

GeoJSON is a text file format for storing geographic shapes. Open any `.geojson`
file in a text editor and you will see something like:

```json
{"type": "FeatureCollection", "features": [{"geometry": {"coordinates": [[144.97, -37.80], ...]}}]}
```

Each pair of numbers is one corner point of the boundary polygon. The ABS publishes
boundary files in this format for SA2, LGA, and other boundaries, you can download
them from the ABS website or use your own.

The cells below use `FITZROY_COLLINGWOOD`, a small inner-Melbourne area used as the
worked example. To use your own boundary, update the path in the setup cell and
replace `FITZROY_COLLINGWOOD` with your variable name in the cells below.

### Load and inspect the Fitzroy/Collingwood boundary

Run the cell below to open the file and print its contents. You do not need to
understand all of it; just notice that coordinates come in pairs and that the
numbers look like Melbourne longitude/latitude values.

In [ ]:
# Open the GeoJSON file and read its contents
with open(FITZROY_COLLINGWOOD) as f:
    fitzroy_geojson = json.load(f)  # json.load converts the text file into a Python dictionary

# Get the geometry (the boundary shape) of the first feature in the file
feature = fitzroy_geojson['features'][0]
geom = feature['geometry']

# Get the outer ring of the polygon (the boundary itself, not any holes inside it)
ring = geom['coordinates'][0]

print(f'Boundary type: {geom["type"]}')
print(f'Number of corner points: {len(ring)}')
print()
print('First 3 coordinate pairs (GeoJSON format: [longitude, latitude]):')
for coord in ring[:3]:
    print(f'  [lon={coord[0]:.6f}, lat={coord[1]:.6f}]')

---
## The Coordinate Order Problem

GeoJSON stores coordinates as `[longitude, latitude]`, with longitude first.
The Domain API expects `{"lat": ..., "lon": ...}`, with latitude first as named fields.

If you send coordinates in the wrong order, the polygon will be placed somewhere in the
Indian Ocean. The API will not warn you, it will simply return zero results, and the
probe credit is spent.

The `geojson_file_to_api_polygon()` function handles the conversion automatically. But
if you ever build coordinates manually or load them from a different source, the swap is
easy to introduce silently.

`validate_australia_coords()` is a zero-credit guard imported from `utils.py` that
checks whether your converted points fall within Australia's bounding box before you
make any API call. The cell below the conversion demonstrates it on both correct and
swapped coordinates.

In [ ]:
# Convert the GeoJSON file to the format the Domain API expects
api_polygon = geojson_file_to_api_polygon(FITZROY_COLLINGWOOD)

print(f'Number of polygon points for the API: {len(api_polygon)}')
print()
print('First 3 points after conversion (API format: {lat, lon}):')
for pt in api_polygon[:3]:
    print(f'  lat={pt["lat"]:.6f}, lon={pt["lon"]:.6f}')
print()
print('Compare: the lat and lon values are the same but the order is now lat-first.')

# This is the complete geoWindow payload structure ready to send to the API
polygon_geo_window = {'polygon': {'points': api_polygon}}

In [ ]:
# validate_australia_coords is defined in utils.py and imported above.
# Run it after any coordinate conversion and before the first probe.

# --- correct coordinates (should pass) ---
print('=== Correct coordinates (after conversion) ===')
validate_australia_coords(api_polygon, label='Fitzroy/Collingwood polygon')

# --- swapped coordinates (GeoJSON order sent directly) ---
print()
print('=== Swapped coordinates (GeoJSON [lon, lat] sent as-is) ===')
swapped = [{'lat': pt['lon'], 'lon': pt['lat']} for pt in api_polygon]
result = validate_australia_coords(swapped, label='Fitzroy/Collingwood polygon (swapped)')
# result is False when coordinates fail the check -- the warning above explains what to fix

---
## Bounding Box vs. Polygon: Which Should You Use?

The API supports two spatial filter types:

**Bounding box** (`geoWindow.box`): a rectangle defined by four edges (north, south,
east, west). Think of drawing a rectangle around the boundary on a map. It is simple
to compute but will include listings from outside your actual study area. Any
property in the corners of the rectangle but outside the true boundary will be
captured.

**Polygon** (`geoWindow.polygon`): traces the exact boundary. Only listings whose
property coordinates fall inside the polygon are returned. This is more precise
but requires sending more coordinates in each request.

**Rule of thumb:** use the polygon for official boundaries (SA2, LGA) where
precision matters. Use the bounding box for quick exploratory queries or for
very irregular shapes with hundreds of vertices (where the polygon payload
becomes very large).

### Compute the bounding box

The `geojson_bbox_to_api_box()` function finds the northernmost, southernmost,
easternmost, and westernmost points of the polygon.

In [ ]:
# Compute the bounding box of the Fitzroy/Collingwood polygon
api_box = geojson_bbox_to_api_box(FITZROY_COLLINGWOOD)

print('Bounding box corners:')
print(f'  North edge (topLeft lat):     {api_box["topLeft"]["lat"]:.6f}')
print(f'  South edge (bottomRight lat): {api_box["bottomRight"]["lat"]:.6f}')
print(f'  West edge  (topLeft lon):     {api_box["topLeft"]["lon"]:.6f}')
print(f'  East edge  (bottomRight lon): {api_box["bottomRight"]["lon"]:.6f}')

# This is the complete geoWindow payload for the bounding box
box_geo_window = {'box': api_box}

### Compare the record counts: bounding box vs. polygon

The next cell runs a 1-credit probe for each approach. **This costs 2 credits total.**
The difference in counts is the number of listings captured by the rectangle's corners
that fall outside the true polygon boundary.

Note: the test boundary file used in this notebook is a simple rectangle (5 points), so
the tight bounding box computed above is identical to the polygon and would return the
same count, which defeats the purpose of the demonstration. To illustrate the
difference clearly, the comparison below uses a deliberately padded bounding box that
extends 0.015 degrees beyond each edge of the polygon, making it meaningfully larger
than the true boundary.

In [ ]:
# Define the base search criteria (same for both approaches)
base_criteria = {
    'listingType': 'Sold',
    'listedSince': '2021-01-01',
}

# Padded bounding box: extends 0.015 degrees beyond each edge of the polygon.
# This is deliberate -- the tight bbox computed from a rectangular polygon is
# identical to the polygon itself, so padding is needed to show a meaningful difference.
PAD = 0.015
padded_box_geo_window = {
    'box': {
        'topLeft':     {'lat': api_box['topLeft']['lat']     + PAD,
                        'lon': api_box['topLeft']['lon']     - PAD},
        'bottomRight': {'lat': api_box['bottomRight']['lat'] - PAD,
                        'lon': api_box['bottomRight']['lon'] + PAD},
    }
}

# Probe with padded bounding box (1 credit)
payload_box = {**base_criteria, 'geoWindow': padded_box_geo_window}
count_box = probe_count(payload_box, tracker)
tracker.checkpoint('Bounding box probe')

# Probe with polygon (1 credit)
payload_polygon = {**base_criteria, 'geoWindow': polygon_geo_window}
count_polygon = probe_count(payload_polygon, tracker)
tracker.checkpoint('Polygon probe')

print(f'Padded bounding box count: {count_box:,}')
print(f'Polygon count:             {count_polygon:,}')

if count_box is not None and count_polygon is not None:
    overshoot = count_box - count_polygon
    print(f'Overshoot: {overshoot:,} extra listings captured by the box but outside the true boundary')

**Reading the result:** the padded bounding box returns more listings than the polygon because it extends beyond the true boundary. Those extra listings belong to neighbouring streets and suburbs caught by the rectangle's edges. With a real SA2 or LGA boundary (which has an irregular shape), the tight bounding box would also produce overshoot, often substantial in areas with irregular coastlines or diagonal edges. For a precise study, always use the polygon.

---
## Fetching Listings Within the Polygon

The probe showed over 3,500 records for the Fitzroy/Collingwood polygon since 2021, so cursor advancement is needed. The `fetch_all_cursor` function below is the same one introduced in Notebook 3, the only difference from a suburb-based query is that the payload uses `geoWindow` instead of `locations`. The function itself is identical.

**Note on credits:** each page of 200 records costs 1 credit. The cell below will spend roughly 18–20 credits to fetch the full dataset.

In [ ]:
def fetch_all_cursor(base_payload, start_date, page_size=200):
    """Retrieve ALL listings for a query that may exceed 1,000 results.
    Works with both suburb-based and spatial (geoWindow) queries.
    """
    all_listings = {}
    current_since = start_date
    prev_cursor = None
    run = 0

    while True:
        run += 1
        page = 1
        batch = []
        hit_limit = False

        while True:
            payload = {
                **base_payload,
                'listedSince': current_since,
                'sort': {'sortKey': 'DateListed', 'direction': 'Ascending'},
                'pageSize': page_size,
                'pageNumber': page,
            }
            r = tracker.post(
                f'{PROXY_BASE}/v1/listings/residential/_search',
                json_body=payload,
            )
            if r.status_code == 400 and 'Cannot page beyond 1000' in r.text:
                hit_limit = True
                break
            if r.status_code != 200:
                print(f'  Run {run}, page {page}: HTTP {r.status_code}')
                return list(all_listings.values())
            groups = r.json()
            for g in groups:
                raw = g.get('listings') or g.get('listing')
                if isinstance(raw, list):
                    batch.extend(raw)
                elif isinstance(raw, dict):
                    batch.append(raw)
            actual_page_size = int(r.headers.get('x-pagination-pagesize', page_size))
            if len(groups) < actual_page_size:
                break
            page += 1

        before = len(all_listings)
        for item in batch:
            if item.get('id'):
                all_listings[item['id']] = item
        print(f'  Run {run}: +{len(all_listings) - before:>5} new | total: {len(all_listings):>6}')

        if not hit_limit or not batch:
            break

        dates = [item['dateListed'][:10] for item in batch if item.get('dateListed')]
        if not dates:
            break
        cursor = max(dates)
        if cursor == prev_cursor:
            print(f'  Cursor stuck at {cursor}. Stopping.')
            break
        prev_cursor = cursor
        current_since = cursor

    return list(all_listings.values())


print('Fetching Fitzroy/Collingwood polygon listings (cursor advancement)...')
polygon_raw = fetch_all_cursor(payload_polygon, start_date='2021-01-01')
tracker.checkpoint('Polygon fetch')

# Extract coordinates and key fields
polygon_rows = []
for item in polygon_raw:
    prop = item.get('propertyDetails') or {}
    lat = prop.get('latitude')
    lon = prop.get('longitude')
    if lat and lon:
        polygon_rows.append({
            'id':          item.get('id'),
            'suburb':      prop.get('suburb'),
            'lat':         lat,
            'lon':         lon,
            'date_listed': (item.get('dateListed') or '')[:10],
        })

df_polygon = pd.DataFrame(polygon_rows)
print(f'\nListings with coordinates: {len(df_polygon):,}')
df_polygon.head()

---
## Visualising the Difference: Polygon vs. Bounding Box

The map below plots the listings already fetched (`df_polygon`) over an OpenStreetMap base layer, with both boundaries drawn on top:

- **Blue line**: the polygon boundary (the true study area)
- **Red line**: the padded bounding box (extends beyond the polygon in all directions)
- **Blue dots**: listings returned by the polygon query

You can see visually that the red box extends into neighbouring streets and suburbs. Any properties in that fringe would be captured by a bounding box query but not a polygon query.

> If the map appears blank with a "notebook not trusted" warning, go to **Edit → Trust Notebook** in the Jupyter menu and re-run the cell.

In [ ]:
centre_lat = (api_box['topLeft']['lat'] + api_box['bottomRight']['lat']) / 2
centre_lon = (api_box['topLeft']['lon'] + api_box['bottomRight']['lon']) / 2
m = folium.Map(location=[centre_lat, centre_lon], zoom_start=14)

# Polygon boundary (blue)
folium.Polygon(
    locations=[[pt['lat'], pt['lon']] for pt in api_polygon],
    color='blue', fill=False, weight=2, tooltip='Polygon boundary',
).add_to(m)

# Padded bounding box boundary (red)
tl = padded_box_geo_window['box']['topLeft']
br = padded_box_geo_window['box']['bottomRight']
folium.Polygon(
    locations=[[tl['lat'], tl['lon']], [tl['lat'], br['lon']],
               [br['lat'], br['lon']], [br['lat'], tl['lon']],
               [tl['lat'], tl['lon']]],
    color='red', fill=False, weight=2, tooltip='Padded bounding box',
).add_to(m)

# Listings from the polygon fetch (blue dots)
for _, row in df_polygon.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lon']], radius=3,
        color='steelblue', fill=True, fill_opacity=0.5,
    ).add_to(m)

display(m)

---
## What If Something Goes Wrong?

**Boundary file not found:**
Update the `FITZROY_COLLINGWOOD` path in the setup cell to the actual location of your file. The error message will print the full path it tried so you can see exactly what to correct. If you pasted the path from Windows file manager, replace each single backslash with a forward slash (`/`) or a doubled backslash (`\\`), otherwise Python cannot read it.

**Probe returns 0 records from the polygon:**
This usually means the coordinates are in the wrong order (longitude and latitude
swapped). The `geojson_file_to_api_polygon()` function handles the conversion
automatically, so this should not happen if you use it. If you provide your own
coordinates manually, double-check the order and run `validate_australia_coords()` before the first probe.

**The map shows dots in the wrong location (e.g. the ocean):**
Longitude and latitude may have been swapped somewhere. Melbourne is at roughly
lat=-37.8, lon=144.9. If your dots appear at lat=144.9, lon=-37.8 (which is in
the Indian Ocean), the coordinate order needs to be fixed.

**The folium map appears blank with a "notebook not trusted" warning:**
Go to **Edit → Trust Notebook** in the Jupyter menu and re-run the cell.

**Authentication failure:**
Check your `.env` file and confirm `AURIN_USERNAME` and `AURIN_PASSWORD` are set correctly. See the [Data Access Guide](../../phase-1/01-data-access-guide.md) if you need to set up credentials from scratch.

---

## You Have Reached the End of Phase 2

You can now query the Domain API by suburb name or by a custom geographic boundary, and retrieve all matching records regardless of how many there are.

Here is a summary of what each notebook covered:

- **Notebook 0:** Setting up your environment and making your first API call.
- **Notebook 1:** The listings endpoint, filtering, pagination, and saving results to CSV.
- **Notebook 2:** The suburb performance statistics endpoint, quarterly trends, and charting.
- **Notebook 3:** Cursor advancement for queries over 1,000 records, plus checkpoint recovery for long-running extractions.
- **Notebook 4 (this notebook):** Spatial queries using GeoJSON boundary files.

From here, apply these tools to your own research question. The [Credit Calculator guide](../../phase-1/03-credit-calculator.md) can help you estimate costs before running a large extraction.

---
## Credit Summary

In [ ]:
tracker.summary()